# Sandbox Inception — run the SDK inside a sandbox

A powerful pattern: run the Python SDK **inside** a sandbox to create and manage *other* sandboxes. This is how AI agents, CI pipelines, and orchestrators that themselves run inside a sandbox can spin up their own isolated child environments — with **no secrets**, authenticating with the sandbox group's **managed identity**.

The flow:
1. Create a sandbox group with a **system-assigned managed identity**.
2. Grant that identity the **Container Apps SandboxGroup Data Owner** role so it can manage sandboxes.
3. Boot an **orchestrator** sandbox, seeding it with environment variables for zero-config auth.
4. Install the SDK inside the orchestrator and write a spawn script.
5. Run it — from *inside* the sandbox, the SDK uses `ManagedIdentityCredential` to create and manage a **child** sandbox.

## Prerequisites
- An Azure subscription where you can **create role assignments** (Owner or User Access Administrator) at the resource group scope.
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively) (`az login`).
- The `azure-containerapps-sandbox` SDK installed in this notebook's Python environment.

Click **Run All** to execute the full flow, or step through each cell to inspect outputs along the way.


In [ ]:
# Install the SDK and supporting packages (skip if already installed)
%pip install -q --upgrade azure-containerapps-sandbox azure-identity azure-mgmt-resource azure-mgmt-authorization

### 0. Initialize variables and clients

We pull the current Azure subscription from the CLI and build the control-plane clients up front:

- `resource_mgmt_client` — manages the Azure **resource group**.
- `sandboxgroup_mgmt_client` — ARM control-plane client for the **sandbox group** (and its managed identity).
- `auth_client` — assigns the data-plane **role** to the managed identity.

The data-plane `SandboxGroupClient` is built at the end of step 1, once we know the sandbox group's region.

Adjust `resource_group_name`, `sandbox_group_name`, and `location` if you want to target existing resources or a different region.


In [ ]:
import json, subprocess, sys, time, uuid, textwrap
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.containerapps.sandbox import SandboxGroupManagementClient, SandboxGroupClient, endpoint_for_region

# ── Configuration ─────────────────────────────────────────────────────────
lab_name                = Path(globals().get('__vsc_ipynb_file__')).stem
resource_group_name     = 'aca-sandboxes-rg'                       # change if using existing Resource Group
sandbox_group_name      = f'lab-{lab_name}-{uuid.uuid4().hex[:6]}' # unique per run
location                = 'westus3'                                # change to your preferred Azure Location
lab_labels              = {'lab': lab_name}                        # tag every resource so cleanup can find them

# SDK wheel installed *inside* the orchestrator sandbox (early-access preview build).
sdk_wheel_url = ('https://github.com/microsoft/azure-container-apps/releases/download/'
                 'python-sdk-v0.1.0b1-early-access/'
                 'azure_containerapps_sandbox-0.1.0b1-py3-none-any.whl')

# ── Get current subscription info from Azure CLI (run `az login` first) ───
try:
    proc = subprocess.run('az account show -o json',
        capture_output=True, text=True, check=True, shell=True)
    subscription = json.loads(proc.stdout)
    subscription_id = subscription['id']
except subprocess.CalledProcessError as e:
    sys.exit(f'❌ Azure CLI not installed or not signed in. Run `az login` first.\n{e.stderr}')

cli_credential = AzureCliCredential() # reused by every Azure client below
resource_mgmt_client = ResourceManagementClient(cli_credential, subscription_id) # manage resource groups
sandboxgroup_mgmt_client = SandboxGroupManagementClient(cli_credential, subscription_id=subscription_id, resource_group=resource_group_name) # manage sandbox groups
auth_client = AuthorizationManagementClient(cli_credential, subscription_id) # assign data-plane role to the managed identity

print(f'🧪 Lab:                   {lab_name}')
print(f'👤 User:                  {subscription["user"]["name"]}')
print(f'🏛️ Tenant Id:             {subscription["tenantId"]}')
print(f'🔑 Subscription:          {subscription["name"]} ({subscription_id})')

print(f'🌍 Target Azure Region:   {location}')
print(f'📁 Target Resource Group: {resource_group_name}')
print(f'📦 Target Sandbox Group:  {sandbox_group_name}')

### 1. Create the sandbox group with a managed identity

A **sandbox group** is the top-level management boundary — all sandboxes, disk images, snapshots, volumes, and secrets are scoped to it.

For inception we give the group a **system-assigned managed identity**. Code running *inside* a sandbox in this group can then call back to the control plane as that identity (`ManagedIdentityCredential`) — no `az login`, no secrets. We capture the identity's `principalId` for the role assignment in the next step.

This cell creates (or reuses) the resource group, then creates the sandbox group with `identity={'type': 'SystemAssigned'}`, and finally builds the data-plane `SandboxGroupClient`.


In [ ]:
if resource_mgmt_client.resource_groups.check_existence(resource_group_name):
    resource_group = resource_mgmt_client.resource_groups.get(resource_group_name)
    print(f'♻️ Using existing resource group: {resource_group.name} ({resource_group.location})')
else:
    resource_group = resource_mgmt_client.resource_groups.create_or_update(
        resource_group_name, {'location': location, 'tags': lab_labels})
    print(f'✅ Created resource group: {resource_group.name} ({resource_group.location})')

# Reuse the sandbox group if it exists; otherwise create it with a system-assigned managed identity.
existing = next((g for g in sandboxgroup_mgmt_client.list_groups() if g.name == sandbox_group_name), None)
if existing:
    action, icon = 'Using existing', '♻️'
else:
    sandboxgroup_mgmt_client.begin_create_group(
        sandbox_group_name, location=resource_group.location,
        identity={'type': 'SystemAssigned'}, tags=lab_labels).result()
    action, icon = 'Created', '✅'

# Refetch to get a fully-hydrated object. The managed identity's principalId can take a
# few seconds to populate, so re-read once if it's not there yet.
sandboxgroup = sandboxgroup_mgmt_client.get_group(sandbox_group_name)
identity = sandboxgroup.identity or {}
mi_principal_id = identity.get('principalId')
if not mi_principal_id:
    time.sleep(5)
    sandboxgroup = sandboxgroup_mgmt_client.get_group(sandbox_group_name)
    identity = sandboxgroup.identity or {}
    mi_principal_id = identity.get('principalId')

sandbox_group_url = f'https://sandboxes.azure.com/sandbox-groups/{subscription_id}/{resource_group_name}/{sandbox_group_name}/sandboxes'
print(f'{icon} {action} sandbox group: {sandboxgroup.name}')
print(f'  🆔 Id:                   {sandboxgroup.id}')
print(f'  🌍 Location:             {sandboxgroup.location}')
print(f'  ⚙️ Provisioning state:   {sandboxgroup.properties["provisioningState"]}')
print(f'  🪪 Identity type:        {identity.get("type")}')
print(f'  🔑 MI principalId:       {mi_principal_id}')
print(f'  🔗 Portal:               {sandbox_group_url}')

assert mi_principal_id, 'Managed identity principalId not available yet — re-run this cell.'

# Build the data-plane client now that we know the group's region — used to create the
# orchestrator sandbox (sandboxes.azure.com, regional ADC endpoint).
client = SandboxGroupClient(
    endpoint_for_region(sandboxgroup.location), cli_credential,
    subscription_id=subscription_id, resource_group=resource_group_name,
    sandbox_group=sandbox_group_name)
print(f'  🔌 Data-plane endpoint:  {endpoint_for_region(sandboxgroup.location)}')


### 2. Grant yourself data-plane access

Before this notebook can create the orchestrator sandbox, **you** (the signed-in user) need the data-plane role on the sandbox group. We assign **Container Apps SandboxGroup Data Owner** at the resource group scope to your own identity.

This is separate from the managed identity grant in the next step: this one lets *the notebook host* create sandboxes; the next one lets *code running inside a sandbox* do the same.

> Role assignments can take 30–60 seconds to propagate. The SDK's HTTP pipeline retries `403`s during this window.


In [ ]:
# Grant YOURSELF (the signed-in user) the data-plane role so this notebook can create the
# orchestrator sandbox. Without this, the data-plane SandboxGroupClient calls would return 403.
role_name = 'Container Apps SandboxGroup Data Owner'
scope = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}' # can be as narrow as a single sandbox group
role_def = next(auth_client.role_definitions.list(scope, filter=f"roleName eq '{role_name}'"))

# Resolve the signed-in user's object id from the Azure CLI.
proc = subprocess.run('az ad signed-in-user show --query id -o tsv',
    capture_output=True, text=True, check=True, shell=True)
user_principal_id = proc.stdout.strip()

try:
    auth_client.role_assignments.create(scope, uuid.uuid4(), {
        'role_definition_id': role_def.id,
        'principal_id': user_principal_id,
        'principal_type': 'User',
    })
    print(f"✅ Assigned '{role_name}' to you ({subscription['user']['name']})")
    print('  ⏳ May take 30–60s to propagate before you can create sandboxes.')
except Exception as ex:
    if 'RoleAssignmentExists' in str(ex) or 'Conflict' in str(ex):
        print(f"♻️ '{role_name}' already assigned to you ({subscription['user']['name']})")
    else:
        raise

print(f'  🆔 Role definition: {role_def.id}')
print(f'  👤 User principalId: {user_principal_id}')
print(f'  🎯 Scope:           {scope}')


### 3. Grant the managed identity data-plane access

The managed identity exists, but it can't *do* anything until we grant it a role. We assign **Container Apps SandboxGroup Data Owner** at the resource group scope so the identity can create and manage sandboxes in this group.

This is what makes inception work: once the orchestrator sandbox runs as this identity, it has exactly the same data-plane permissions we have here — without ever handling a credential.

> Role assignments can take 30–60 seconds to propagate. The SDK's HTTP pipeline retries `403`s during this window.


In [ ]:
# Grant the sandbox group's managed identity the data-plane role, so code running INSIDE a
# sandbox (via ManagedIdentityCredential) can create and manage sibling sandboxes.
role_name = 'Container Apps SandboxGroup Data Owner'
scope = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}' # the scope can be as narrow as an individual sandbox group; just needs to include the sandbox group resource ID
role_def = next(auth_client.role_definitions.list(scope, filter=f"roleName eq '{role_name}'"))

try:
    auth_client.role_assignments.create(scope, uuid.uuid4(), {
        'role_definition_id': role_def.id,
        'principal_id': mi_principal_id,
        'principal_type': 'ServicePrincipal',
    })
    print(f"✅ Assigned '{role_name}' to the managed identity")
    print('  ⏳ May take 30–60s to propagate before the orchestrator can use it.')
except Exception as ex:
    if 'RoleAssignmentExists' in str(ex) or 'Conflict' in str(ex):
        print(f"♻️ '{role_name}' already assigned to the managed identity")
    else:
        raise

print(f'  🆔 Role definition: {role_def.id}')
print(f'  🎯 Scope:           {scope}')


### 4. Create the orchestrator sandbox

Now boot the **orchestrator** — an ordinary sandbox that will run the SDK to spawn children. We seed it with environment variables (`AZURE_SUBSCRIPTION_ID`, `ACA_RESOURCE_GROUP`, `ACA_SANDBOX_GROUP`, `ACA_SANDBOXGROUP_REGION`) so the SDK inside it can connect with **zero configuration**.

Inside the sandbox, `ManagedIdentityCredential()` automatically picks up the group's managed identity — there's nothing to log in to and no secret to inject.


In [ ]:
# Boot the orchestrator sandbox. The environment variables let the SDK authenticate with
# zero config from inside the sandbox via ManagedIdentityCredential — no az login, no secrets.
orchestrator = client.begin_create_sandbox(
    disk='python-3.14',
    labels=lab_labels,
    environment={
        'AZURE_SUBSCRIPTION_ID': subscription_id,
        'ACA_RESOURCE_GROUP': resource_group_name,
        'ACA_SANDBOX_GROUP': sandbox_group_name,
        'ACA_SANDBOXGROUP_REGION': sandboxgroup.location,
    },
).result()
orchestrator.wait_for_running(timeout=120)

details = client.get_sandbox(orchestrator.sandbox_id)
print(f'📦 Orchestrator sandbox running: {orchestrator.sandbox_id}  state={details.state}')
print(f"   🐍 python: {orchestrator.exec('python3 --version').stdout.strip()}")


### 5. Install the SDK and write the spawn script inside the orchestrator

With the orchestrator running, we `pip install` the SDK *inside* it and write a small `spawn.py`. That script is the inception payload: it builds a `SandboxGroupClient` using `ManagedIdentityCredential` and the seeded env vars, then creates a **child** sandbox, runs a command in it, and deletes it.

Note that the script contains **no credentials** — it relies entirely on the managed identity we wired up in steps 1–3.


In [ ]:
# Install the SDK inside the orchestrator (needs egress to the wheel URL + PyPI for azure-identity).
print('📥 Installing the SDK inside the orchestrator sandbox...')
install = orchestrator.exec(f'pip install --quiet {sdk_wheel_url} azure-identity')
print(f'   🔢 pip exit code: {install.exit_code}')
if install.exit_code != 0:
    raise RuntimeError(install.stderr[-2000:] or install.stdout[-2000:])

# Write the inception payload: from INSIDE the sandbox, use the managed identity to create a child.
spawn_script = textwrap.dedent('''
    import os
    from azure.identity import ManagedIdentityCredential
    from azure.containerapps.sandbox import SandboxGroupClient, endpoint_for_region

    # No secrets: the credential is the sandbox group's managed identity.
    credential = ManagedIdentityCredential()
    client = SandboxGroupClient(
        endpoint_for_region(os.environ["ACA_SANDBOXGROUP_REGION"]),
        credential,
        subscription_id=os.environ["AZURE_SUBSCRIPTION_ID"],
        resource_group=os.environ["ACA_RESOURCE_GROUP"],
        sandbox_group=os.environ["ACA_SANDBOX_GROUP"],
    )

    # Create a child sandbox from inside the parent, run a command, then clean it up.
    child = client.begin_create_sandbox(disk="ubuntu", labels={"role": "child"}).result()
    child.wait_for_running(timeout=120)
    result = child.exec("echo hello from child sandbox")
    print(result.stdout.strip())
    child.delete()
    client.close()
''')
orchestrator.write_file('/tmp/spawn.py', spawn_script)
print('📝 Wrote /tmp/spawn.py inside the orchestrator')


### 6. Run it — the orchestrator spawns a child sandbox

Execute `spawn.py` inside the orchestrator. From within that sandbox, the SDK authenticates as the managed identity and creates a **child** sandbox in the same group, runs a command in it, and tears it down.

Seeing `hello from child sandbox` in the orchestrator's stdout confirms the full inception loop worked — a sandbox created and managed another sandbox, with no secrets anywhere.


In [ ]:
# Run the script — from inside the sandbox, the SDK authenticates via managed identity and
# creates (then deletes) a child sandbox. No `az login` and no secrets are involved.
print('🚀 Running /tmp/spawn.py inside the orchestrator (this creates a child sandbox)...')
run = orchestrator.exec('python3 /tmp/spawn.py')
print(f'  🔢 Exit code: {run.exit_code}')

if run.stdout.strip():
    print('\n✅ Orchestrator stdout:')
    print(run.stdout.strip())
if run.stderr.strip():
    print('\n⚠️ Orchestrator stderr:')
    print(run.stderr.strip()[-2000:])



### 7. Clean up

We use the **lab label** (`lab_labels`) attached to the orchestrator to find and delete it in one pass — so leftover sandboxes from a previous run (or a crashed kernel) get cleaned up too. The child sandbox already deletes itself inside `spawn.py`.

Then we remove the **role assignment** we created for the managed identity and delete the **sandbox group**, which cascades and removes any remaining sandboxes scoped to it.

The resource group delete is left commented out so you don't accidentally remove a shared RG — uncomment it if you created the RG just for this lab.


In [ ]:
# Delete every sandbox tagged with this lab's label (the orchestrator + any leftovers).
# The child sandbox already deleted itself inside spawn.py.
lab_sandboxes = list(client.list_sandboxes(labels=lab_labels))
print(f'🔎 Found {len(lab_sandboxes)} sandbox(es) tagged {lab_labels}')
for s in lab_sandboxes:
    client.delete_sandbox(s.id)
    print(f'  🗑️ Deleted sandbox: {s.id}')

# Remove the role assignment we created for the managed identity.
try:
    for ra in auth_client.role_assignments.list_for_scope(
            scope, filter=f"principalId eq '{mi_principal_id}'"):
        if ra.role_definition_id == role_def.id:
            auth_client.role_assignments.delete_by_id(ra.id)
            print(f'🗑️ Removed role assignment: {ra.name}')
except Exception as ex:
    print(f'⚠️ Could not remove role assignment: {ex}')

# Delete the sandbox group — cascades to any remaining sandboxes scoped to it.
sandboxgroup_mgmt_client.delete_group(sandbox_group_name)
print(f'🗑️ Deleted sandbox group: {sandbox_group_name}')

# Optionally delete the resource group (fire-and-forget — runs in the background).
# resource_mgmt_client.resource_groups.begin_delete(resource_group_name)
# print(f'🗑️ Deleting resource group (async): {resource_group_name}')
